# Aegis Command - Zero-Shot Model Bake-Off (Colab GPU Pipeline)

This notebook executes the full **Zero-Shot Model Bake-Off** on Google Colab (T4 / V100 / A100 GPU) across all 155 human-verified aerial frames.

### Ground Truth Sample Sizes:
- **Total Frames**: 155 frames (1920x1080 resolution)
- **Person**: $n = 1,737$ ground truth boxes (Primary target)
- **Truck**: $n = 153$ ground truth boxes
- **Car**: $n = 58$ ground truth boxes (*Note: n=58, unreliable sample size*)
- **Bus**: $n = 0$ ground truth boxes (*Excluded from report as meaningless*)
- **Total Boxes**: 1,948 certified boxes

### Model & Resolution Matrix:
- **Architectures**: `yolov8n`, `yolov8s`, `yolov8m`, `yolo11n`, `yolo11s`, `yolo11m`
- **Input Sizes**: `640`, `960`, `1280` px
- **Tiling**: Full frame + SAHI (Sliced Aided Hyper Inference, 640x640 slices, 0.2 overlap)
- **Primary Metric**: `person` AP50 with 95% Bootstrap Confidence Intervals (1,000 resamples of frames)
- **Operating Points**: Precision & Recall evaluated at confidence thresholds `0.25` and `0.35`
- **False Positive Rate**: Total False Positives per frame (FP / 155)

In [ ]:
# 1. Verify GPU Allocation
!nvidia-smi

In [ ]:
# 2. Install Required Dependencies
%pip install -q ultralytics sahi albumentations opencv-python matplotlib pandas numpy tabulate

In [ ]:
# 3. Clone Aegis Command Repository
!git clone https://github.com/TanmayJain-dev/Aegis-Command.git
%cd Aegis-Command
!git checkout feat/accuracy-v2
!git pull

In [ ]:
# 4. Strict Certification & Integrity Gate Check
from pathlib import Path
import sys
from vision.eval.eval_detection import check_verification_gate

labels_path = Path("vision/eval/labels")
marker = check_verification_gate(labels_path)
print(f"[GATE PASSED] Verified marker found at {marker}")
print("Ground truth labels verified with cryptographic SHA256 integrity.")

In [ ]:
# 5. Execute Zero-Shot Matrix Evaluation Runner
import os
import json
import time
from vision.eval.eval_detection import run_evaluation

# Define evaluation matrix
MODELS = ["yolov8n.pt", "yolov8s.pt", "yolov8m.pt", "yolo11n.pt", "yolo11s.pt", "yolo11m.pt"]
SIZES = [640, 960, 1280]

matrix_results = []
os.makedirs("docs/results", exist_ok=True)

for model_name in MODELS:
    for imgsz in SIZES:
        print(f"\n{'='*70}")
        print(f"RUNNING ZERO-SHOT EVALUATION: {model_name} @ {imgsz}px (GPU)")
        print(f"{'='*70}")
        out_file = f"docs/results/eval_{model_name.replace('.pt', '')}_{imgsz}px.json"
        res = run_evaluation(
            images_dir="vision/eval/images",
            labels_dir="vision/eval/labels",
            model_path=model_name,
            imgsz=imgsz,
            conf=0.25,
            split="all",
            device="cuda:0",
            out_results=out_file
        )
        matrix_results.append({
            "model": model_name,
            "imgsz": imgsz,
            "results": res
        })

with open("docs/results/zero_shot_bakeoff.json", "w", encoding="utf-8") as f:
    json.dump(matrix_results, f, indent=2)

print("\n[COMPLETE] All zero-shot matrix combinations evaluated successfully.")

In [ ]:
# 6. SAHI (Sliced Aided Hyper Inference) GPU Evaluation
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from pathlib import Path
import numpy as np
from vision.eval.metrics import calculate_metrics_for_class, bootstrap_metric_ci
from vision.eval.eval_detection import load_yolo_labels, CLASS_NAMES

print("Running SAHI Sliced Inference with yolov8s (slice=640x640, overlap=0.2)...")
detection_model = AutoDetectionModel.from_pretrained(
    model_type="yolov8",
    model_path="yolov8s.pt",
    confidence_threshold=0.20,
    device="cuda:0"
)

images_path = Path("vision/eval/images")
images = sorted(list(images_path.glob("*.jpg")) + list(images_path.glob("*.png")))
image_ids = [img.stem for img in images]
labels_path = Path("vision/eval/labels")
ground_truths = load_yolo_labels(labels_path, image_ids)

sahi_preds = []
latencies = []

for img in images:
    t0 = time.perf_counter()
    result = get_sliced_prediction(
        str(img),
        detection_model,
        slice_height=640,
        slice_width=640,
        overlap_height_ratio=0.2,
        overlap_width_ratio=0.2,
        verbose=0
    )
    latencies.append((time.perf_counter() - t0) * 1000.0)

    # Standard COCO to Eval mapping: person=0, car=2->1, truck=7->2
    COCO_MAP = {0: 0, 2: 1, 7: 2}
    for obj in result.object_prediction_list:
        cid = obj.category.id
        if cid in COCO_MAP:
            bbox = obj.bbox.to_xyxy()
            sahi_preds.append({
                "image_id": img.stem,
                "class": COCO_MAP[cid],
                "conf": float(obj.score.value),
                "box": [bbox[0]/1920.0, bbox[1]/1080.0, bbox[2]/1920.0, bbox[3]/1080.0]
            })

person_preds = [p for p in sahi_preds if p["class"] == 0 and p["conf"] >= 0.25]
m_person = calculate_metrics_for_class(person_preds, ground_truths, target_class=0)
ci = bootstrap_metric_ci(person_preds, ground_truths, image_ids, target_class=0)

print(f"SAHI Person AP50: {m_person['ap50']:.4f} [95% CI: {ci['ap50_ci'][0]:.4f} - {ci['ap50_ci'][1]:.4f}]")
print(f"SAHI Person Recall: {m_person['recall']:.4f} [95% CI: {ci['recall_ci'][0]:.4f} - {ci['recall_ci'][1]:.4f}]")
print(f"SAHI Mean Latency: {np.mean(latencies):.2f} ms/frame")

In [ ]:
# 7. Render Formatted Markdown Benchmark Comparison Table
import pandas as pd

rows = []
for item in matrix_results:
    m_name = item["model"].replace(".pt", "")
    sz = item["imgsz"]
    r = item["results"]
    lat = r["latency_ms"]["mean_ms"]
    ci = r["bootstrap_ci_95_person"]
    p_025 = r["metrics_at_conf_0.25"]["person"]
    p_035 = r["metrics_at_conf_0.35"]["person"]
    c_025 = r["metrics_at_conf_0.25"]["car"]
    t_025 = r["metrics_at_conf_0.25"]["truck"]

    rows.append({
        "Model": m_name,
        "Size (px)": sz,
        "Person AP50 [95% CI]": f"{p_025['ap50']:.4f} [{ci['ap50_ci'][0]:.3f}, {ci['ap50_ci'][1]:.3f}]",
        "Person Recall [95% CI]": f"{p_025['recall']:.4f} [{ci['recall_ci'][0]:.3f}, {ci['recall_ci'][1]:.3f}]",
        "Person P@0.25": p_025['precision'],
        "Person R@0.25": p_025['recall'],
        "Person P@0.35": p_035['precision'],
        "Person R@0.35": p_035['recall'],
        "Person FP/frame": p_025['fp_per_frame'],
        "Truck AP50": t_025['ap50'],
        "Car AP50 (n=58)": f"{c_025['ap50']:.4f}*",
        "Latency (ms)": lat
    })

df = pd.DataFrame(rows)
print("### ZERO-SHOT MODEL BAKE-OFF COMPARISON MATRIX")
print(df.to_markdown(index=False))
print("\n* Note: Car sample size is n=58 (unreliable). Bus is excluded (n=0).")

In [ ]:
# 8. Download Results JSON
from google.colab import files
files.download("docs/results/zero_shot_bakeoff.json")